In [ ]:
# ============================================================================
# MBP PREDICTOR GUI - Google Colab Version
# Microtubule-Binding Protein Prediction with Gradio Interface
# ============================================================================

# Install required packages
!pip install gradio lightgbm biopython transformers torch joblib -q

import gradio as gr
import pandas as pd
import numpy as np
import joblib  # Changed from pickle to joblib
import os
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/MBP_PREDICTOR')

# Import your feature extraction modules
import sys
sys.path.append('/content/drive/MyDrive/MBP_PREDICTOR')

from src.feature_aac import compute_aac
from src.feature_charge import compute_charge
from src.feature_hydrophobicity import compute_hydrophobicity
from src.feature_pi import compute_pi
from src.feature_motif import compute_motif
from src.feature_embeddings import ProteinEmbedder

print("✅ All modules imported successfully!")

# ============================================================================
# LOAD MODELS
# ============================================================================

print("\n" + "="*60)
print("LOADING TRAINED MODELS")
print("="*60)

# Load both models using joblib
try:
    model_92 = joblib.load('TRAINED_MODELS/LightGBM_92.pkl')
    print("✅ LightGBM_92 model loaded successfully")
    print(f"   Type: {type(model_92)}")
    print(f"   Features: {model_92.n_features_}")
    print(f"   Classes: {model_92.classes_}")
except Exception as e:
    print(f"❌ Error loading LightGBM_92: {e}")
    model_92 = None

try:
    model_91 = joblib.load('TRAINED_MODELS/LightGBM_91.22.pkl')
    print("✅ LightGBM_91.22 model loaded successfully")
    print(f"   Type: {type(model_91)}")
    print(f"   Features: {model_91.n_features_}")
    print(f"   Classes: {model_91.classes_}")
except Exception as e:
    print(f"❌ Error loading LightGBM_91.22: {e}")
    model_91 = None

# Initialize embedder
print("\nInitializing ProtBERT embedder...")
try:
    embedder = ProteinEmbedder()
    print("✅ Embedder loaded successfully")
except Exception as e:
    print(f"❌ Error loading embedder: {e}")
    embedder = None

# ============================================================================
# FEATURE EXTRACTION FUNCTION
# ============================================================================

def extract_features(sequence):
    """Extract all features from a protein sequence"""
    try:
        features = {}

        # AAC features (20)
        features.update(compute_aac(sequence))

        # Charge feature (1)
        features.update(compute_charge(sequence))

        # Hydrophobicity features (3)
        features.update(compute_hydrophobicity(sequence))

        # pI feature (1)
        pi_value = compute_pi(sequence)
        features["pI"] = float(pi_value) if pi_value is not None else 7.0

        # Motif features (6)
        features.update(compute_motif(sequence))

        # Embeddings (1024)
        if embedder is not None:
            try:
                embedding_vector = embedder.get_embedding(sequence)
                embedding_features = {f"E_{i}": float(val) for i, val in enumerate(embedding_vector)}
                features.update(embedding_features)
            except Exception as e:
                print(f"Warning: Embedding failed, using zeros: {e}")
                embedding_features = {f"E_{i}": 0.0 for i in range(1024)}
                features.update(embedding_features)
        else:
            # No embedder available
            embedding_features = {f"E_{i}": 0.0 for i in range(1024)}
            features.update(embedding_features)

        return features

    except Exception as e:
        print(f"Error extracting features: {e}")
        return None

# ============================================================================
# VALIDATION FUNCTION
# ============================================================================

def validate_sequence(sequence):
    """Validate protein sequence"""
    # Clean sequence
    seq_clean = sequence.upper().replace(" ", "").replace("\n", "").replace("\t", "")

    # Check if empty
    if not seq_clean:
        return False, "Sequence is empty", None

    # Check length
    if len(seq_clean) < 10:
        return False, "Sequence too short (minimum 10 amino acids)", None

    if len(seq_clean) > 1000:
        return False, "Sequence too long (maximum 1000 amino acids)", None

    # Check valid amino acids
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    invalid_chars = set(seq_clean) - valid_aa

    if invalid_chars:
        return False, f"Invalid amino acids detected: {', '.join(sorted(invalid_chars))}", None

    return True, "Valid sequence", seq_clean

# ============================================================================
# PREDICTION FUNCTIONS
# ============================================================================

def predict_single_sequence(sequence, model_choice):
    """Predict for a single sequence"""

    # Validate sequence
    is_valid, message, seq_clean = validate_sequence(sequence)

    if not is_valid:
        return f"❌ Error: {message}", None, None, None

    # Select model
    if model_choice == "LightGBM_92 (92.0% Accuracy)":
        model = model_92
        model_name = "LightGBM_92"
    else:
        model = model_91
        model_name = "LightGBM_91.22"

    if model is None:
        return "❌ Error: Selected model not loaded", None, None, None

    # Extract features
    print(f"Extracting features for sequence of length {len(seq_clean)}...")
    features = extract_features(seq_clean)

    if features is None:
        return "❌ Error: Feature extraction failed", None, None, None

    # Convert to DataFrame
    feature_df = pd.DataFrame([features])

    # Make prediction
    try:
        prediction = model.predict(feature_df)[0]
        probability = model.predict_proba(feature_df)[0]

        mbp_prob = probability[1] * 100
        non_mbp_prob = probability[0] * 100

        # Create result message
        if prediction == 1:
            result = f"✅ **MICROTUBULE-BINDING PROTEIN (MBP)**"
            confidence = mbp_prob
            emoji = "🟢"
        else:
            result = f"❌ **NON-MICROTUBULE-BINDING PROTEIN**"
            confidence = non_mbp_prob
            emoji = "🔴"

        details = f"""
{emoji} **Prediction Result**

**Classification:** {"Microtubule-Binding Protein" if prediction == 1 else "Non-Microtubule-Binding Protein"}
**Confidence:** {confidence:.2f}%

**Probabilities:**
- MBP: {mbp_prob:.2f}%
- Non-MBP: {non_mbp_prob:.2f}%

**Sequence Details:**
- Length: {len(seq_clean)} amino acids
- Model: {model_name}
- Features Extracted: {len(features)}
"""

        return details, mbp_prob, non_mbp_prob, seq_clean

    except Exception as e:
        return f"❌ Error during prediction: {str(e)}", None, None, None

def predict_batch_csv(csv_file, model_choice):
    """Predict for multiple sequences from CSV"""

    if csv_file is None:
        return "❌ Please upload a CSV file", None

    try:
        # Read CSV
        df = pd.read_csv(csv_file.name)

        # Check for required columns
        if 'Sequence' not in df.columns:
            return "❌ Error: CSV must contain a 'Sequence' column", None

        # Optional Protein_ID column
        if 'Protein_ID' not in df.columns:
            df['Protein_ID'] = [f"Protein_{i+1}" for i in range(len(df))]

        # Select model
        if model_choice == "LightGBM_92 (92.0% Accuracy)":
            model = model_92
            model_name = "LightGBM_92"
        else:
            model = model_91
            model_name = "LightGBM_91.22"

        if model is None:
            return "❌ Error: Selected model not loaded", None

        # Process each sequence
        results = []
        print(f"\nProcessing {len(df)} sequences...")

        for idx, row in df.iterrows():
            seq = row['Sequence']
            protein_id = row['Protein_ID']

            # Validate
            is_valid, message, seq_clean = validate_sequence(seq)

            if not is_valid:
                results.append({
                    'Protein_ID': protein_id,
                    'Sequence': seq[:50] + "..." if len(seq) > 50 else seq,
                    'Prediction': 'Invalid',
                    'MBP_Probability': 0,
                    'Non_MBP_Probability': 0,
                    'Error': message
                })
                continue

            # Extract features
            features = extract_features(seq_clean)

            if features is None:
                results.append({
                    'Protein_ID': protein_id,
                    'Sequence': seq_clean[:50] + "..." if len(seq_clean) > 50 else seq_clean,
                    'Prediction': 'Error',
                    'MBP_Probability': 0,
                    'Non_MBP_Probability': 0,
                    'Error': 'Feature extraction failed'
                })
                continue

            # Predict
            try:
                feature_df = pd.DataFrame([features])
                prediction = model.predict(feature_df)[0]
                probability = model.predict_proba(feature_df)[0]

                results.append({
                    'Protein_ID': protein_id,
                    'Sequence': seq_clean[:50] + "..." if len(seq_clean) > 50 else seq_clean,
                    'Sequence_Length': len(seq_clean),
                    'Prediction': 'MBP' if prediction == 1 else 'Non-MBP',
                    'MBP_Probability': round(probability[1] * 100, 2),
                    'Non_MBP_Probability': round(probability[0] * 100, 2),
                    'Confidence': round(max(probability) * 100, 2)
                })

            except Exception as e:
                results.append({
                    'Protein_ID': protein_id,
                    'Sequence': seq_clean[:50] + "..." if len(seq_clean) > 50 else seq_clean,
                    'Prediction': 'Error',
                    'MBP_Probability': 0,
                    'Non_MBP_Probability': 0,
                    'Error': str(e)
                })

            # Progress update
            if (idx + 1) % 10 == 0:
                print(f"Processed {idx + 1}/{len(df)} sequences...")

        # Create results DataFrame
        results_df = pd.DataFrame(results)

        # Save results
        output_path = "predictions_output.csv"
        results_df.to_csv(output_path, index=False)

        # Summary statistics
        mbp_count = (results_df['Prediction'] == 'MBP').sum()
        non_mbp_count = (results_df['Prediction'] == 'Non-MBP').sum()
        error_count = results_df['Prediction'].isin(['Error', 'Invalid']).sum()

        summary = f"""
✅ **Batch Prediction Complete**

**Summary:**
- Total Sequences: {len(results_df)}
- Predicted as MBP: {mbp_count}
- Predicted as Non-MBP: {non_mbp_count}
- Errors/Invalid: {error_count}

**Model Used:** {model_name}

**Results saved to:** {output_path}
"""

        return summary, results_df

    except Exception as e:
        return f"❌ Error processing CSV: {str(e)}", None

# ============================================================================
# GRADIO INTERFACE
# ============================================================================

# Custom CSS for better styling
custom_css = """
.gradio-container {
    font-family: 'Arial', sans-serif;
}
.gr-button-primary {
    background: linear-gradient(90deg, #667eea 0%, #764ba2 100%) !important;
    border: none !important;
}
"""

# Create Gradio Interface with Tabs
with gr.Blocks(css=custom_css, title="MBP Predictor") as demo:

    gr.Markdown("""
    # 🧬 Microtubule-Binding Protein (MBP) Predictor

    ### Advanced ML-based prediction system for identifying microtubule-binding proteins

    **Features:**
    - 🎯 Two high-accuracy LightGBM models (92% and 91.22% accuracy)
    - 📊 1055 molecular features including ProtBERT embeddings
    - 🔬 Single sequence or batch prediction support
    - 📁 CSV upload for high-throughput analysis

    ---
    """)

    # Model selection (shared across tabs)
    model_selector = gr.Radio(
        choices=["LightGBM_92 (92.0% Accuracy)", "LightGBM_91.22 (91.22% Accuracy)"],
        value="LightGBM_92 (92.0% Accuracy)",
        label="Select Model"
    )

    # Create tabs
    with gr.Tabs():

        # Tab 1: Single Sequence Prediction
        with gr.Tab("🔬 Single Sequence Prediction"):
            gr.Markdown("### Enter a protein sequence for prediction")

            with gr.Row():
                with gr.Column():
                    sequence_input = gr.Textbox(
                        label="Protein Sequence",
                        placeholder="Enter amino acid sequence (e.g., MKTIIALSYIFCLVFA...)",
                        lines=5
                    )

                    with gr.Row():
                        example_mbp = gr.Button("Load MBP Example", size="sm")
                        example_non_mbp = gr.Button("Load Non-MBP Example", size="sm")
                        clear_btn = gr.Button("Clear", size="sm")

                    predict_btn = gr.Button("🚀 Predict", variant="primary", size="lg")

                with gr.Column():
                    result_output = gr.Markdown(label="Prediction Result")

                    with gr.Row():
                        mbp_prob_output = gr.Number(label="MBP Probability (%)", precision=2)
                        non_mbp_prob_output = gr.Number(label="Non-MBP Probability (%)", precision=2)

                    sequence_display = gr.Textbox(label="Cleaned Sequence", lines=3)

            # Example sequences
            example_mbp.click(
                fn=lambda: "KRIVQRIKDFLRNLVPRTES",
                outputs=sequence_input
            )

            example_non_mbp.click(
                fn=lambda: "AGLQFPVGRVHRLLRK",
                outputs=sequence_input
            )

            clear_btn.click(
                fn=lambda: "",
                outputs=sequence_input
            )

            # Prediction
            predict_btn.click(
                fn=predict_single_sequence,
                inputs=[sequence_input, model_selector],
                outputs=[result_output, mbp_prob_output, non_mbp_prob_output, sequence_display]
            )

        # Tab 2: Batch CSV Prediction
        with gr.Tab("📁 Batch CSV Prediction"):
            gr.Markdown("""
            ### Upload a CSV file with multiple sequences

            **Required CSV Format:**
            - Must contain a column named `Sequence` with protein sequences
            - Optional: `Protein_ID` column for sequence identifiers
            - Example:

            | Protein_ID | Sequence |
            |------------|----------|
            | Protein_1  | MKTII... |
            | Protein_2  | AGLQF... |
            """)

            with gr.Row():
                with gr.Column():
                    csv_input = gr.File(
                        label="Upload CSV File",
                        file_types=[".csv"]
                    )

                    batch_predict_btn = gr.Button("🚀 Predict Batch", variant="primary", size="lg")

                with gr.Column():
                    batch_result = gr.Markdown(label="Batch Prediction Summary")

            batch_output_df = gr.Dataframe(
                label="Prediction Results",
                wrap=True
            )

            # Batch prediction
            batch_predict_btn.click(
                fn=predict_batch_csv,
                inputs=[csv_input, model_selector],
                outputs=[batch_result, batch_output_df]
            )

    gr.Markdown("""
    ---

    ### 📊 Model Information

    **Training Dataset:**
    - 3,072 protein sequences
    - Balanced dataset (1,533 MBP, 1,539 Non-MBP)
    - Sequence length: 10-1000 amino acids

    **Features:**
    - Amino Acid Composition (20 features)
    - Charge properties (1 feature)
    - Hydrophobicity (3 features)
    - Isoelectric point (1 feature)
    - Motif patterns (6 features)
    - ProtBERT embeddings (1024 features)

    **Performance:**
    - LightGBM_92: 92.0% accuracy
    - LightGBM_91.22: 91.22% accuracy

    ---
    *Developed for microtubule-binding protein research*
    """)

# Launch the interface
print("\n" + "="*60)
print("LAUNCHING GRADIO INTERFACE")
print("="*60)

demo.launch(share=True, debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All modules imported successfully!

LOADING TRAINED MODELS
✅ LightGBM_92 model loaded successfully
   Type: <class 'lightgbm.sklearn.LGBMClassifier'>
   Features: 1055
   Classes: [0 1]
✅ LightGBM_91.22 model loaded successfully
   Type: <class 'lightgbm.sklearn.LGBMClassifier'>
   Features: 1055
   Classes: [0 1]

Initializing ProtBERT embedder...
Loading model Rostlab/prot_bert on cuda...
✅ Embedder loaded successfully

LAUNCHING GRADIO INTERFACE
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://aac28d1c33b6929a3d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Extracting features for sequence of length 972...
Extracting features for sequence of length 468...


In [2]:
# ============================================================================
# MBP PREDICTOR GUI - Google Colab Version with Sequence Analysis
# Microtubule-Binding Protein Prediction with Gradio Interface
# ============================================================================

# Install required packages
!pip install gradio lightgbm biopython transformers torch joblib matplotlib seaborn -q

import gradio as gr
import pandas as pd
import numpy as np
import joblib
import os
from google.colab import drive
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO
import base64
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/MBP_PREDICTOR')

# Import your feature extraction modules
import sys
sys.path.append('/content/drive/MyDrive/MBP_PREDICTOR')

from src.feature_aac import compute_aac
from src.feature_charge import compute_charge
from src.feature_hydrophobicity import compute_hydrophobicity
from src.feature_pi import compute_pi
from src.feature_motif import compute_motif
from src.feature_embeddings import ProteinEmbedder

print("All modules imported successfully!")

# ============================================================================
# LOAD MODELS
# ============================================================================

print("\n" + "="*60)
print("LOADING TRAINED MODELS")
print("="*60)

# Load both models using joblib
try:
    model_92 = joblib.load('TRAINED_MODELS/LightGBM_92.pkl')
    print("LightGBM_92 model loaded successfully")
    print(f"   Type: {type(model_92)}")
    print(f"   Features: {model_92.n_features_}")
    print(f"   Classes: {model_92.classes_}")
except Exception as e:
    print(f"Error loading LightGBM_92: {e}")
    model_92 = None

try:
    model_91 = joblib.load('TRAINED_MODELS/LightGBM_91.22.pkl')
    print("LightGBM_91.22 model loaded successfully")
    print(f"   Type: {type(model_91)}")
    print(f"   Features: {model_91.n_features_}")
    print(f"   Classes: {model_91.classes_}")
except Exception as e:
    print(f"Error loading LightGBM_91.22: {e}")
    model_91 = None

# Initialize embedder
print("\nInitializing ProtBERT embedder...")
try:
    embedder = ProteinEmbedder()
    print("Embedder loaded successfully")
except Exception as e:
    print(f"Error loading embedder: {e}")
    embedder = None

# Load training statistics for comparison
print("\nLoading training statistics...")
try:
    training_stats = pd.read_csv("training_data_properties.csv")
    mbp_stats = training_stats[training_stats['Label'] == 1]
    non_mbp_stats = training_stats[training_stats['Label'] == 0]
    print("Training statistics loaded")
except:
    print("Warning: Training statistics not found")
    mbp_stats = None
    non_mbp_stats = None

# ============================================================================
# FEATURE EXTRACTION FUNCTION
# ============================================================================

def extract_features(sequence):
    """Extract all features from a protein sequence"""
    try:
        features = {}

        # AAC features (20)
        features.update(compute_aac(sequence))

        # Charge feature (1)
        features.update(compute_charge(sequence))

        # Hydrophobicity features (3)
        features.update(compute_hydrophobicity(sequence))

        # pI feature (1)
        pi_value = compute_pi(sequence)
        features["pI"] = float(pi_value) if pi_value is not None else 7.0

        # Motif features (6)
        features.update(compute_motif(sequence))

        # Embeddings (1024)
        if embedder is not None:
            try:
                embedding_vector = embedder.get_embedding(sequence)
                embedding_features = {f"E_{i}": float(val) for i, val in enumerate(embedding_vector)}
                features.update(embedding_features)
            except Exception as e:
                print(f"Warning: Embedding failed, using zeros: {e}")
                embedding_features = {f"E_{i}": 0.0 for i in range(1024)}
                features.update(embedding_features)
        else:
            embedding_features = {f"E_{i}": 0.0 for i in range(1024)}
            features.update(embedding_features)

        return features

    except Exception as e:
        print(f"Error extracting features: {e}")
        return None

# ============================================================================
# SEQUENCE ANALYSIS FUNCTION
# ============================================================================

def analyze_sequence(sequence, features):
    """Generate detailed sequence analysis with visualizations"""

    # Calculate key properties
    seq_length = len(sequence)
    lysine_content = features['AAC_K'] * 100
    arginine_content = features['AAC_R'] * 100
    kr_content = lysine_content + arginine_content
    net_charge = features['Charge']
    pi_value = features['pI']
    hydro_mean = features['Hydro_mean']

    # Count motifs
    motif_count = sum([features[f'Motif_{m}'] for m in ['KR', 'RR', 'KK', 'KXK', 'K[RK]K', 'R..R']])

    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Sequence Biochemical Analysis', fontsize=14, fontweight='bold')

    # Plot 1: Amino Acid Composition (top 10)
    aac_features = {k: v*100 for k, v in features.items() if k.startswith('AAC_')}
    top_aa = sorted(aac_features.items(), key=lambda x: x[1], reverse=True)[:10]
    aa_names = [aa[0].replace('AAC_', '') for aa in top_aa]
    aa_values = [aa[1] for aa in top_aa]

    axes[0, 0].bar(aa_names, aa_values, color='steelblue', edgecolor='black')
    axes[0, 0].set_xlabel('Amino Acid')
    axes[0, 0].set_ylabel('Composition (%)')
    axes[0, 0].set_title('Top 10 Amino Acid Composition')
    axes[0, 0].tick_params(axis='x', rotation=45)

    # Plot 2: Key MBP indicators vs Training averages
    if mbp_stats is not None and non_mbp_stats is not None:
        properties = ['Lysine+Arginine\n(%)', 'Net Charge', 'pI']
        test_values = [kr_content, net_charge, pi_value]
        mbp_avg = [
            (mbp_stats['Lysine'].mean() + mbp_stats['Arginine'].mean()) * 100,
            mbp_stats['Net_Charge'].mean(),
            mbp_stats['pI'].mean()
        ]
        non_mbp_avg = [
            (non_mbp_stats['Lysine'].mean() + non_mbp_stats['Arginine'].mean()) * 100,
            non_mbp_stats['Net_Charge'].mean(),
            non_mbp_stats['pI'].mean()
        ]

        x = np.arange(len(properties))
        width = 0.25

        axes[0, 1].bar(x - width, test_values, width, label='Test Sequence', color='red', edgecolor='black')
        axes[0, 1].bar(x, mbp_avg, width, label='MBP Average', color='green', alpha=0.7, edgecolor='black')
        axes[0, 1].bar(x + width, non_mbp_avg, width, label='Non-MBP Average', color='gray', alpha=0.7, edgecolor='black')

        axes[0, 1].set_xlabel('Property')
        axes[0, 1].set_ylabel('Value')
        axes[0, 1].set_title('Comparison with Training Data')
        axes[0, 1].set_xticks(x)
        axes[0, 1].set_xticklabels(properties)
        axes[0, 1].legend()
    else:
        axes[0, 1].text(0.5, 0.5, 'Training statistics\nnot available',
                       ha='center', va='center', fontsize=12)
        axes[0, 1].set_title('Comparison with Training Data')

    # Plot 3: Charge distribution along sequence
    charges = []
    for aa in sequence:
        if aa in ['K', 'R', 'H']:
            charges.append(1)
        elif aa in ['D', 'E']:
            charges.append(-1)
        else:
            charges.append(0)

    # Running average
    window = min(20, len(charges)//5) if len(charges) > 5 else len(charges)
    if window > 0:
        running_charge = pd.Series(charges).rolling(window=window, center=True).mean()
        axes[1, 0].plot(running_charge, color='purple', linewidth=2)
        axes[1, 0].axhline(y=0, color='black', linestyle='--', linewidth=1)
        axes[1, 0].set_xlabel('Position')
        axes[1, 0].set_ylabel('Charge (rolling avg)')
        axes[1, 0].set_title(f'Charge Distribution (window={window})')
        axes[1, 0].fill_between(range(len(running_charge)), running_charge, 0,
                                 where=(running_charge > 0), alpha=0.3, color='blue', label='Positive')
        axes[1, 0].fill_between(range(len(running_charge)), running_charge, 0,
                                 where=(running_charge < 0), alpha=0.3, color='red', label='Negative')
        axes[1, 0].legend()

    # Plot 4: Key metrics summary
    axes[1, 1].axis('off')
    summary_text = f"""
    SEQUENCE PROPERTIES

    Length: {seq_length} amino acids

    Basic Amino Acids:
      Lysine (K): {lysine_content:.2f}%
      Arginine (R): {arginine_content:.2f}%
      K+R Total: {kr_content:.2f}%

    Charge Properties:
      Net Charge: {net_charge:.2f}
      Isoelectric Point: {pi_value:.2f}

    Hydrophobicity: {hydro_mean:.3f}

    MBP-related Motifs: {motif_count}
    """

    axes[1, 1].text(0.1, 0.9, summary_text, fontsize=11,
                   verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()

    # Save plot to buffer
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    plt.close()

    return buf

# ============================================================================
# VALIDATION FUNCTION
# ============================================================================

def validate_sequence(sequence):
    """Validate protein sequence"""
    seq_clean = sequence.upper().replace(" ", "").replace("\n", "").replace("\t", "")

    if not seq_clean:
        return False, "Sequence is empty", None

    if len(seq_clean) < 10:
        return False, "Sequence too short (minimum 10 amino acids)", None

    if len(seq_clean) > 1000:
        return False, "Sequence too long (maximum 1000 amino acids)", None

    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    invalid_chars = set(seq_clean) - valid_aa

    if invalid_chars:
        return False, f"Invalid amino acids detected: {', '.join(sorted(invalid_chars))}", None

    return True, "Valid sequence", seq_clean

# ============================================================================
# PREDICTION FUNCTIONS
# ============================================================================

def predict_single_sequence(sequence, model_choice):
    """Predict for a single sequence with analysis"""

    is_valid, message, seq_clean = validate_sequence(sequence)

    if not is_valid:
        return f"Error: {message}", None, None, None, None

    # Select model
    if model_choice == "LightGBM_92 (92.0% Accuracy)":
        model = model_92
        model_name = "LightGBM_92"
    else:
        model = model_91
        model_name = "LightGBM_91.22"

    if model is None:
        return "Error: Selected model not loaded", None, None, None, None

    # Extract features
    print(f"Extracting features for sequence of length {len(seq_clean)}...")
    features = extract_features(seq_clean)

    if features is None:
        return "Error: Feature extraction failed", None, None, None, None

    # Convert to DataFrame
    feature_df = pd.DataFrame([features])

    # Make prediction
    try:
        prediction = model.predict(feature_df)[0]
        probability = model.predict_proba(feature_df)[0]

        mbp_prob = probability[1] * 100
        non_mbp_prob = probability[0] * 100

        # Create result message
        if prediction == 1:
            result = "MICROTUBULE-BINDING PROTEIN (MBP)"
            confidence = mbp_prob
            status = "POSITIVE"
        else:
            result = "NON-MICROTUBULE-BINDING PROTEIN"
            confidence = non_mbp_prob
            status = "NEGATIVE"

        details = f"""
**Prediction Result**

**Status:** {status}
**Classification:** {result}
**Confidence:** {confidence:.2f}%

**Probabilities:**
- MBP: {mbp_prob:.2f}%
- Non-MBP: {non_mbp_prob:.2f}%

**Model:** {model_name}
**Features Extracted:** {len(features)}
**Sequence Length:** {len(seq_clean)} amino acids
"""

        # Generate analysis plot
        analysis_plot = analyze_sequence(seq_clean, features)

        return details, mbp_prob, non_mbp_prob, seq_clean, analysis_plot

    except Exception as e:
        return f"Error during prediction: {str(e)}", None, None, None, None

def predict_batch_csv(csv_file, model_choice):
    """Predict for multiple sequences from CSV"""

    if csv_file is None:
        return "Please upload a CSV file", None

    try:
        df = pd.read_csv(csv_file.name)

        if 'Sequence' not in df.columns:
            return "Error: CSV must contain a 'Sequence' column", None

        if 'Protein_ID' not in df.columns:
            df['Protein_ID'] = [f"Protein_{i+1}" for i in range(len(df))]

        if model_choice == "LightGBM_92 (92.0% Accuracy)":
            model = model_92
            model_name = "LightGBM_92"
        else:
            model = model_91
            model_name = "LightGBM_91.22"

        if model is None:
            return "Error: Selected model not loaded", None

        results = []
        print(f"\nProcessing {len(df)} sequences...")

        for idx, row in df.iterrows():
            seq = row['Sequence']
            protein_id = row['Protein_ID']

            is_valid, message, seq_clean = validate_sequence(seq)

            if not is_valid:
                results.append({
                    'Protein_ID': protein_id,
                    'Sequence': seq[:50] + "..." if len(seq) > 50 else seq,
                    'Prediction': 'Invalid',
                    'MBP_Probability': 0,
                    'Non_MBP_Probability': 0,
                    'Error': message
                })
                continue

            features = extract_features(seq_clean)

            if features is None:
                results.append({
                    'Protein_ID': protein_id,
                    'Sequence': seq_clean[:50] + "..." if len(seq_clean) > 50 else seq_clean,
                    'Prediction': 'Error',
                    'MBP_Probability': 0,
                    'Non_MBP_Probability': 0,
                    'Error': 'Feature extraction failed'
                })
                continue

            try:
                feature_df = pd.DataFrame([features])
                prediction = model.predict(feature_df)[0]
                probability = model.predict_proba(feature_df)[0]

                results.append({
                    'Protein_ID': protein_id,
                    'Sequence': seq_clean[:50] + "..." if len(seq_clean) > 50 else seq_clean,
                    'Sequence_Length': len(seq_clean),
                    'Prediction': 'MBP' if prediction == 1 else 'Non-MBP',
                    'MBP_Probability': round(probability[1] * 100, 2),
                    'Non_MBP_Probability': round(probability[0] * 100, 2),
                    'Confidence': round(max(probability) * 100, 2)
                })

            except Exception as e:
                results.append({
                    'Protein_ID': protein_id,
                    'Sequence': seq_clean[:50] + "..." if len(seq_clean) > 50 else seq_clean,
                    'Prediction': 'Error',
                    'MBP_Probability': 0,
                    'Non_MBP_Probability': 0,
                    'Error': str(e)
                })

            if (idx + 1) % 10 == 0:
                print(f"Processed {idx + 1}/{len(df)} sequences...")

        results_df = pd.DataFrame(results)

        output_path = "predictions_output.csv"
        results_df.to_csv(output_path, index=False)

        mbp_count = (results_df['Prediction'] == 'MBP').sum()
        non_mbp_count = (results_df['Prediction'] == 'Non-MBP').sum()
        error_count = results_df['Prediction'].isin(['Error', 'Invalid']).sum()

        summary = f"""
**Batch Prediction Complete**

**Summary:**
- Total Sequences: {len(results_df)}
- Predicted as MBP: {mbp_count}
- Predicted as Non-MBP: {non_mbp_count}
- Errors/Invalid: {error_count}

**Model Used:** {model_name}

**Results saved to:** {output_path}
"""

        return summary, results_df

    except Exception as e:
        return f"Error processing CSV: {str(e)}", None

# ============================================================================
# GRADIO INTERFACE
# ============================================================================

custom_css = """
.gradio-container {
    font-family: 'Arial', sans-serif;
}
.gr-button-primary {
    background: linear-gradient(90deg, #667eea 0%, #764ba2 100%) !important;
    border: none !important;
}
"""

with gr.Blocks(css=custom_css, title="MBP Predictor") as demo:

    gr.Markdown("""
    # Microtubule-Binding Protein (MBP) Predictor

    ### Advanced ML-based prediction system for identifying microtubule-binding proteins

    **Features:**
    - Two high-accuracy LightGBM models (92% and 91.22% accuracy)
    - 1055 molecular features including ProtBERT embeddings
    - Detailed sequence analysis with visualizations
    - Single sequence or batch prediction support
    - CSV upload for high-throughput analysis

    ---
    """)

    model_selector = gr.Radio(
        choices=["LightGBM_92 (92.0% Accuracy)", "LightGBM_91.22 (91.22% Accuracy)"],
        value="LightGBM_92 (92.0% Accuracy)",
        label="Select Model"
    )

    with gr.Tabs():

        with gr.Tab("Single Sequence Prediction"):
            gr.Markdown("### Enter a protein sequence for prediction")

            with gr.Row():
                with gr.Column():
                    sequence_input = gr.Textbox(
                        label="Protein Sequence",
                        placeholder="Enter amino acid sequence (e.g., MKTIIALSYIFCLVFA...)",
                        lines=5
                    )

                    with gr.Row():
                        example_mbp = gr.Button("Load MBP Example", size="sm")
                        example_non_mbp = gr.Button("Load Non-MBP Example", size="sm")
                        clear_btn = gr.Button("Clear", size="sm")

                    predict_btn = gr.Button("Predict", variant="primary", size="lg")

                with gr.Column():
                    result_output = gr.Markdown(label="Prediction Result")

                    with gr.Row():
                        mbp_prob_output = gr.Number(label="MBP Probability (%)", precision=2)
                        non_mbp_prob_output = gr.Number(label="Non-MBP Probability (%)", precision=2)

                    sequence_display = gr.Textbox(label="Cleaned Sequence", lines=3)

            # Analysis visualization
            gr.Markdown("### Sequence Analysis")
            analysis_plot = gr.Image(label="Biochemical Analysis")

            example_mbp.click(
                fn=lambda: "KRIVQRIKDFLRNLVPRTES",
                outputs=sequence_input
            )

            example_non_mbp.click(
                fn=lambda: "AGLQFPVGRVHRLLRK",
                outputs=sequence_input
            )

            clear_btn.click(
                fn=lambda: "",
                outputs=sequence_input
            )

            predict_btn.click(
                fn=predict_single_sequence,
                inputs=[sequence_input, model_selector],
                outputs=[result_output, mbp_prob_output, non_mbp_prob_output, sequence_display, analysis_plot]
            )

        with gr.Tab("Batch CSV Prediction"):
            gr.Markdown("""
            ### Upload a CSV file with multiple sequences

            **Required CSV Format:**
            - Must contain a column named `Sequence` with protein sequences
            - Optional: `Protein_ID` column for sequence identifiers
            - Example:

            | Protein_ID | Sequence |
            |------------|----------|
            | Protein_1  | MKTII... |
            | Protein_2  | AGLQF... |
            """)

            with gr.Row():
                with gr.Column():
                    csv_input = gr.File(
                        label="Upload CSV File",
                        file_types=[".csv"]
                    )

                    batch_predict_btn = gr.Button("Predict Batch", variant="primary", size="lg")

                with gr.Column():
                    batch_result = gr.Markdown(label="Batch Prediction Summary")

            batch_output_df = gr.Dataframe(
                label="Prediction Results",
                wrap=True
            )

            batch_predict_btn.click(
                fn=predict_batch_csv,
                inputs=[csv_input, model_selector],
                outputs=[batch_result, batch_output_df]
            )

    gr.Markdown("""
    ---

    ### Model Information

    **Training Dataset:**
    - 3,072 protein sequences
    - Balanced dataset (1,533 MBP, 1,539 Non-MBP)
    - Sequence length: 10-1000 amino acids

    **Features:**
    - Amino Acid Composition (20 features)
    - Charge properties (1 feature)
    - Hydrophobicity (3 features)
    - Isoelectric point (1 feature)
    - Motif patterns (6 features)
    - ProtBERT embeddings (1024 features)

    **Performance:**
    - LightGBM_92: 92.0% accuracy
    - LightGBM_91.22: 91.22% accuracy

    ---
    *Developed for microtubule-binding protein research*
    """)

print("\n" + "="*60)
print("LAUNCHING GRADIO INTERFACE")
print("="*60)

demo.launch(share=True, debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All modules imported successfully!

LOADING TRAINED MODELS
LightGBM_92 model loaded successfully
   Type: <class 'lightgbm.sklearn.LGBMClassifier'>
   Features: 1055
   Classes: [0 1]
LightGBM_91.22 model loaded successfully
   Type: <class 'lightgbm.sklearn.LGBMClassifier'>
   Features: 1055
   Classes: [0 1]

Initializing ProtBERT embedder...
Loading model Rostlab/prot_bert on cuda...
Embedder loaded successfully

Loading training statistics...
Training statistics loaded

LAUNCHING GRADIO INTERFACE
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://44f34cce3915f05b0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging 

Extracting features for sequence of length 972...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 745, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2127, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1904, in postprocess_data
    prediction_value = block.postprocess(prediction_value)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/components/image.py", line 238, in postprocess

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://44f34cce3915f05b0e.gradio.live
